# Double pendulum and chaos

*mechanics · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/mechanics/double_pendulum)

The Lagrangian equations of a double pendulum written as $M_{ij}(q)\,\ddot q_j = f_i(q, \dot q)$. Each RK4 stage builds the $2\times 2$ mass-matrix tensor and calls torch.linalg.solve; two almost identical starts drift apart, showing chaos.

**Vector form**

$$\mathsf M(\mathbf q)\,\ddot{\mathbf q} = \mathbf f(\mathbf q, \dot{\mathbf q}),\qquad \mathbf q = (\theta_1, \theta_2)$$

**Index (tensor) form**

$$M_{bij}(q)\,\ddot q_{bj} = f_{bi}(q, \dot q),\qquad E_b = \tfrac12\,\dot q_{bi}M_{bij}\dot q_{bj} + V_b$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $q_{bi}$ | `y[:, 0]` | (B, 2) | angles $(\theta_1, \theta_2)$ of pendulum $b$ from the downward vertical |
| $\dot q_{bi}$ | `y[:, 1]` | (B, 2) | angular velocities |
| $M_{bij}$ | `mass_matrix(q, m1, m2, L1, L2)` | (B, 2, 2) | configuration-dependent mass matrix, off-diagonal $m_2L_1L_2\cos(\theta_1-\theta_2)$ |
| $f_{bi}$ | `forces(q, q_dot, ...)` | (B, 2) | generalised forces: centrifugal coupling + gravity |
| $\ddot q_{bj}$ | `torch.linalg.solve(M, f)` | (B, 2) | solve $M\ddot q = f$ for the whole batch in one call |
| $\dot q_{bi}M_{bij}\dot q_{bj}$ | `einsum('...i,...ij,...j->...', q_dot, M, q_dot)` | (B,) | $i$ and $j$ summed: twice the kinetic energy |

The batch holds two pendulums whose $\theta_2$ differs by $\delta$. Their difference grows roughly exponentially until the motions are unrelated, although the energy of each stays constant to high accuracy (float64 + RK4 with a small step). For small swings the system has two normal modes; with equal masses and lengths $\omega^2 = (2 \mp \sqrt2)\,g/L$. Rods are massless and the pivot frictionless.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/mechanics/double_pendulum/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    m1, m2, L1, L2, g = params["m1"], params["m2"], params["L1"], params["L2"], 9.81
    y0 = tensors.initial_state(params["theta1"], params["theta2"], params["delta"], dev)

    times, traj = solve.solve(y0, m1, m2, L1, L2, g, params["t_max"])
    e = equation.energy(traj, m1, m2, L1, L2, g)  # (T, B)

    omega, _ = equation.small_angle_modes(m1, m2, L1, L2, g)
    print(f"small-angle normal modes: ω = {omega[0]:.3f}, {omega[1]:.3f} rad/s")
    print(f"relative energy drift: {((e[-1] - e[0]) / e[0].abs()).abs().max():.2e}")
    gap = (traj[:, 0, 0] - traj[:, 1, 0]).abs().max(dim=-1).values  # (T,) largest angle difference
    print(f"angle difference: start {gap[0]:.1e} rad, end {gap[-1]:.2f} rad")
    return plot.animate(times, traj, L1, L2, out_dir / "double_pendulum.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/mechanics/double_pendulum/equation.py
r"""Physics → tensor translation.

Two point masses m_1, m_2 on light rods L_1, L_2. With the angles q = (θ_1, θ_2) as generalised
coordinates, the Euler–Lagrange equations take the form

    \mathsf M(\mathbf q) \, \ddot{\mathbf q} = \mathbf f(\mathbf q, \dot{\mathbf q}),

or in index notation, for a batch b of pendulums and coordinates i, j ∈ {1, 2}:

    M_{bij} \, \ddot q_{bj} = f_{bi}          (repeated j ⇒ summed)

with the 2×2 mass matrix (Δ = θ_1 − θ_2)

    M = [[ (m_1 + m_2) L_1^2,        m_2 L_1 L_2 \cos Δ ],
         [ m_2 L_1 L_2 \cos Δ,       m_2 L_2^2          ]]

and the generalised forces (centrifugal + gravity)

    f_1 = -m_2 L_1 L_2 \sin Δ \, \dot θ_2^2 - (m_1 + m_2) g L_1 \sin θ_1
    f_2 =  m_2 L_1 L_2 \sin Δ \, \dot θ_1^2 - m_2 g L_2 \sin θ_2

Each step we build M_{bij} and f_{bi} and let torch.linalg.solve find \ddot q_{bj}: no hand inversion.
The energy E_b = ½ \dot q_{bi} M_{bij} \dot q_{bj} + V_b (i and j summed) is conserved.
"""

import torch


def mass_matrix(q: torch.Tensor, m1: float, m2: float, L1: float, L2: float) -> torch.Tensor:
    """M_bij for angles q of shape (B, 2) → (B, 2, 2)."""
    coupling = m2 * L1 * L2 * torch.cos(q[:, 0] - q[:, 1])  # (B,)
    M = torch.empty(q.shape[0], 2, 2, dtype=q.dtype, device=q.device)
    M[:, 0, 0] = (m1 + m2) * L1**2
    M[:, 0, 1] = coupling
    M[:, 1, 0] = coupling
    M[:, 1, 1] = m2 * L2**2
    return M


def forces(q: torch.Tensor, q_dot: torch.Tensor, m1: float, m2: float, L1: float, L2: float, g: float) -> torch.Tensor:
    """f_bi (centrifugal + gravity) for q, q_dot of shape (B, 2) → (B, 2)."""
    s = m2 * L1 * L2 * torch.sin(q[:, 0] - q[:, 1])  # (B,)
    f1 = -s * q_dot[:, 1] ** 2 - (m1 + m2) * g * L1 * torch.sin(q[:, 0])
    f2 = s * q_dot[:, 0] ** 2 - m2 * g * L2 * torch.sin(q[:, 1])
    return torch.stack([f1, f2], dim=-1)


def rhs(y: torch.Tensor, m1: float, m2: float, L1: float, L2: float, g: float) -> torch.Tensor:
    r"""dy/dt for y of shape (B, 2, 2): y[:, 0] = q, y[:, 1] = \dot q.  Solves M_{bij} \ddot q_{bj} = f_{bi}."""
    q, q_dot = y[:, 0], y[:, 1]
    M = mass_matrix(q, m1, m2, L1, L2)  # (B, 2, 2)
    f = forces(q, q_dot, m1, m2, L1, L2, g)  # (B, 2)
    q_ddot = torch.linalg.solve(M, f)  # (B, 2)
    return torch.stack([q_dot, q_ddot], dim=1)


def energy(y: torch.Tensor, m1: float, m2: float, L1: float, L2: float, g: float) -> torch.Tensor:
    r"""E_b = ½ \dot q_{bi} M_{bij} \dot q_{bj} + V_b.   y: (..., B, 2, 2) → (..., B)"""
    q, q_dot = y[..., 0, :], y[..., 1, :]
    M = mass_matrix(q.reshape(-1, 2), m1, m2, L1, L2).reshape(*q.shape, 2)  # (..., B, 2, 2)
    kinetic = 0.5 * torch.einsum("...i,...ij,...j->...", q_dot, M, q_dot)
    potential = -(m1 + m2) * g * L1 * torch.cos(q[..., 0]) - m2 * g * L2 * torch.cos(q[..., 1])
    return kinetic + potential


def small_angle_modes(m1: float, m2: float, L1: float, L2: float, g: float) -> tuple[torch.Tensor, torch.Tensor]:
    r"""Normal modes near the bottom: M^0_{ij} \ddot q_j = -K_{ij} q_j  ⇒  (M^0)^{-1}_{ik} K_{kj} e_j = ω^2 e_i.

    Returns ω (2,) sorted ascending and the mode shapes e (2, 2), mode n = column n.
    """
    M0 = mass_matrix(torch.zeros(1, 2, dtype=torch.float64), m1, m2, L1, L2)[0]  # (2, 2) at q = 0
    K = torch.diag(torch.tensor([(m1 + m2) * g * L1, m2 * g * L2], dtype=torch.float64))  # (2, 2)
    omega_sq, e = torch.linalg.eig(torch.linalg.solve(M0, K))  # not symmetric ⇒ general eig
    omega_sq, e = omega_sq.real, e.real
    order = torch.argsort(omega_sq)
    return torch.sqrt(omega_sq[order]), e[:, order]


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/mechanics/double_pendulum/tensors.py
"""Build the concrete tensors: two almost identical initial states, stacked in a batch.

We use float64: chaos amplifies every rounding error, so we want those errors as small as possible.
"""

import math

import torch

DTYPE = torch.float64


def initial_state(theta1_deg: float, theta2_deg: float, delta_deg: float, device: torch.device) -> torch.Tensor:
    """y0 of shape (B=2, 2, 2): both pendulums start at rest; the second has θ_2 nudged by delta.

    y0[b, 0, :] = (θ_1, θ_2)     angles q_bi (radians, measured from the downward vertical)
    y0[b, 1, :] = (0, 0)         angular velocities q̇_bi
    """
    angles = torch.tensor(
        [
            [math.radians(theta1_deg), math.radians(theta2_deg)],
            [math.radians(theta1_deg), math.radians(theta2_deg + delta_deg)],
        ],
        dtype=DTYPE,
        device=device,
    )  # (2, 2)
    return torch.stack([angles, torch.zeros_like(angles)], dim=1)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/mechanics/double_pendulum/solve.py
"""Integrate both double pendulums together with 4th-order Runge–Kutta."""

import torch

from problems._common.integrators import integrate

from .equation import rhs


def solve(
    y0: torch.Tensor, m1: float, m2: float, L1: float, L2: float, g: float, t_max: float, dt: float = 0.004
) -> tuple[torch.Tensor, torch.Tensor]:
    """Return (times (T,), trajectory (T, B, 2, 2)).

    Every RK4 stage builds the (B, 2, 2) mass matrix and calls torch.linalg.solve once for the whole batch.
    """
    n_steps = int(round(t_max / dt))
    traj = integrate(lambda y, t: rhs(y, m1, m2, L1, L2, g), y0, dt, n_steps)
    times = torch.arange(n_steps + 1, device=y0.device, dtype=y0.dtype) * dt
    return times, traj


def positions(q: torch.Tensor, L1: float, L2: float) -> tuple[torch.Tensor, torch.Tensor]:
    """Cartesian positions of both bobs from the angles q (..., 2) → two tensors (..., 2) holding (x, y)."""
    bob1 = L1 * torch.stack([torch.sin(q[..., 0]), -torch.cos(q[..., 0])], dim=-1)
    bob2 = bob1 + L2 * torch.stack([torch.sin(q[..., 1]), -torch.cos(q[..., 1])], dim=-1)
    return bob1, bob2


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/mechanics/double_pendulum/plot.py
"""Animate the two double pendulums on top of each other, with the growing gap between them beside."""

from pathlib import Path

import matplotlib.pyplot as plt
import torch

from problems._common.plotting import ACCENT, frame_indices, save_gif

from .solve import positions


def animate(times: torch.Tensor, traj: torch.Tensor, L1: float, L2: float, path: Path, trail: int = 25) -> Path:
    """trail = number of animation frames that the lower bob's tail remembers."""
    t = times.cpu().numpy()  # (T,)
    q = traj[:, :, 0].cpu()  # (T, B, 2)
    bob1, bob2 = (p.numpy() for p in positions(q, L1, L2))  # (T, B, 2) each
    gap = (q[:, 0] - q[:, 1]).norm(dim=-1).clamp_min(1e-12).log10().numpy()  # (T,) log10 distance of the two runs
    frames = frame_indices(len(t), 100)
    reach = L1 + L2

    fig, (ax_swing, ax_trace) = plt.subplots(1, 2, figsize=(7, 4), gridspec_kw={"width_ratios": [1, 1.3]})

    # --- left: the pendulums ---
    ax_swing.set_xlim(-1.1 * reach, 1.1 * reach)
    ax_swing.set_ylim(-1.1 * reach, 1.1 * reach)
    ax_swing.set_aspect("equal", adjustable="box")
    ax_swing.set_xticks([])
    ax_swing.set_yticks([])
    ax_swing.plot([0], [0], "k^", ms=6)
    arms = [ax_swing.plot([], [], "o-", lw=2, ms=6, color=ACCENT[b], alpha=0.85)[0] for b in range(2)]
    tails = [ax_swing.plot([], [], lw=1, color=ACCENT[b], alpha=0.5)[0] for b in range(2)]
    clock = ax_swing.text(0.03, 0.97, "", transform=ax_swing.transAxes, va="top", fontsize=9)

    # --- right: how far apart the two runs are, in powers of ten (a straight rise = exponential growth) ---
    ax_trace.set_xlim(t[0], t[-1])
    ax_trace.set_ylim(max(gap.min(), gap[0] - 2) - 0.5, gap.max() + 0.5)
    ax_trace.set_xlabel("t (s)")
    ax_trace.set_ylabel("log₁₀ |q_A − q_B|  (rad)")
    ax_trace.set_title("two starts differing by a tiny δ in θ₂", fontsize=9)
    trace = ax_trace.plot([], [], lw=1.2, color="#111827")[0]

    def update(step: int):
        k = frames.index(step)
        start = frames[max(0, k - trail)]  # time step where the tail begins
        for b in range(2):
            arms[b].set_data([0, bob1[step, b, 0], bob2[step, b, 0]], [0, bob1[step, b, 1], bob2[step, b, 1]])
            tails[b].set_data(bob2[start : step + 1, b, 0], bob2[start : step + 1, b, 1])
        trace.set_data(t[: step + 1], gap[: step + 1])
        clock.set_text(f"t = {t[step]:.2f} s")

    fig.tight_layout()
    return save_gif(fig, update, frames, path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'theta1': 120.0,
    'theta2': -10.0,
    'delta': 0.01,
    'm1': 1.0,
    'm2': 1.0,
    'L1': 1.0,
    'L2': 1.0,
    't_max': 20.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.mechanics.double_pendulum")]:
    del sys.modules[name]
main = importlib.import_module("problems.mechanics.double_pendulum.main")
problem_dir = Path("problems/mechanics/double_pendulum")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
